In [1]:
%pip install -r "requirements.txt"

from did_functions import *
import pandas as pd
import kagglehub
import geopandas as gpd
import datetime as dt
import sqlite3
import os

Note: you may need to restart the kernel to use updated packages.


C:\Users\madel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Hardcoded Values
- NCIC Codes are taken from data since they overlap with Vision Zero data for the period of interest

In [2]:
## Hardcoded Values depending on analysis

# Vision Zero city codes
codes_to_drop = ['1942', '3711', '4313', '0105']

# column and date indicators
date_col = "collision_date"
start_date = '2010-01-01'
end_date = '2016-12-31'
collision_cols = ['case_id', date_col, 'county_city_location', 'collision_severity', 'county_location', 'population', 'weather_1', 'primary_collision_factor', 'pcf_violation_category', 'hit_and_run', 'type_of_collision', 'road_condition_1', 'road_surface', 'lighting', 'alcohol_involved', 'latitude', 'longitude']

# data paths
crash_path = "alexgude/california-traffic-collision-data-from-switrs"
ncic_path = r"../data/raw_data/NCIC Code Jurisdiction List_04242023 - Sheet1.csv"
final_data_path = r"../data/clean_data/California_Collisions_Clean.csv"

Load data from the online source

In [3]:
# Download latest data
data = kagglehub.dataset_download(crash_path)
db_file_path = os.path.join(data, 'switrs.sqlite')
ncic = load_data(ncic_path)

NCIC Code Jurisdiction List_04242023 - Sheet1.csv converted to dataframe.


## Data Cleaning

### Grab data for synthetic control

In [4]:
# establish sql connection to database path
conn = sqlite3.connect(db_file_path)
# use sql query to pull data
query = f"SELECT {', '.join(collision_cols)} FROM collisions WHERE {date_col} >= '{start_date}' AND {date_col} <= '{end_date}'"
collisions = pd.read_sql_query(query, conn)

### Organize Cities by NCIC Codes

In [5]:
# combine data sets
collisions_coded = pd.merge(collisions, ncic, how='left', left_on='county_city_location', right_on='Code')
# drop vision zero codes from synthetic data
collisions_ca = collisions_coded[~collisions_coded['Code'].isin(codes_to_drop)]

### Label treatment by city and time


In [6]:
collisions_ca[date_col] = pd.to_datetime(collisions_ca[date_col])
collisions_ca['year_month'] = collisions_ca[date_col].dt.to_period('M')

C:\Users\madel\AppData\Local\Temp\ipykernel_22076\3473901048.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  collisions_ca[date_col] = pd.to_datetime(collisions_ca[date_col])
C:\Users\madel\AppData\Local\Temp\ipykernel_22076\3473901048.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  collisions_ca['year_month'] = collisions_ca[date_col].dt.to_period('M')


Remove unnecessary entries from agency column

In [7]:
# Get rid of Sheriff's Departments in agencies
collisions_ca = collisions_ca[~collisions_ca['Agency'].str.contains('Sheriff|Police', na=False)]

In [ ]:
# Find top 18 cities plus San Francisco in terms of crashes by Agency Name
top_cities = collisions_ca.groupby('Agency')['case_id'].count().sort_values(ascending = False)
top_cities = top_cities.head(19)

collisions_ca = collisions_ca[
    collisions_ca['Agency'].isin(top_cities.index)
]


,case_id,collision_date,county_city_location,collision_severity,county_location,population,weather_1,primary_collision_factor,pcf_violation_category,hit_and_run,...,alcohol_involved,latitude,longitude,CntyCode,County,Code,Agency,Start,End,year_month
6,4392010,2010-01-24,1005,fatal,fresno,>250000,clear,vehicle code violation,pedestrian right of way,not hit and run,...,NaN,NaN,NaN,10.0,Fresno County,1005,Fresno,NaN,NaN,2010-01
7,4392011,2010-01-15,1005,fatal,fresno,>250000,fog,vehicle code violation,dui,not hit and run,...,1.0,NaN,NaN,10.0,Fresno County,1005,Fresno,NaN,NaN,2010-01
12,4392016,2010-01-14,1955,fatal,los angeles,100000 to 250000,clear,vehicle code violation,traffic signals and signs,not hit and run,...,NaN,NaN,NaN,19.0,Los Angeles County,1955,Pomona,NaN,NaN,2010-01
20,4392048,2010-02-13,3607,fatal,san bernardino,100000 to 250000,clear,vehicle code violation,speeding,not hit and run,...,NaN,NaN,NaN,36.0,San Bernardino County,3607,Ontario,NaN,NaN,2010-02
23,4392054,2010-02-21,3607,fatal,san bernardino,100000 to 250000,raining,vehicle code violation,dui,not hit and run,...,1.0,NaN,NaN,36.0,San Bernardino County,3607,Ontario,NaN,NaN,2010-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2929177,90018346,2015-09-10,3801,other injury,san francisco,>250000,clear,vehicle code violation,following too closely,not hit and run,...,NaN,37.72681,-122.40187,38.0,San Francisco County,3801,San Francisco,NaN,NaN,2015-09
2929179,90218184,2016-05-07,1005,fatal,fresno,>250000,clear,other than driver,other than driver (or pedestrian),felony,...,NaN,36.74674,-119.82226,10.0,Fresno County,1005,Fresno,NaN,NaN,2016-05
2929184,90326285,2015-12-06,3009,fatal,orange,100000 to 250000,clear,vehicle code violation,dui,not hit and run,...,1.0,33.78883,-117.93185,30.0,Orange County,3009,Garden Grove,NaN,NaN,2015-12
2929185,6606397,2015-09-19,1953,property damage only,los angeles,100000 to 250000,clear,other improper driving,other improper driving,not hit and run,...,NaN,NaN,NaN,19.0,Los Angeles County,1953,Pasadena,NaN,NaN,2015-09


In [14]:
# save final dataset
collisions_ca.to_csv(final_data_path)